# 02 - Character Segmentation (crops Viola-Jones)

Aquest notebook és una extensió del notebook `02_character_segmentation.ipynb` original, adaptat per processar **múltiples crops reals** generats pel detector Viola-Jones (VJ). En el notebook original partíem d'una única matrícula ja ben retallada i alineada; aquí els crops provenen d'un detector que pot introduir lleugeres **rotacions** i **falsos positius**.

La tècnica de segmentació és exactament la mateixa que l'original:
**`adaptiveThreshold` → `findContours` → filtres geomètrics** sobre bounding boxes.

Afegim dos mecanismes nous:
1. **Correcció d'inclinació (deskew)** prèvia a la binarització, basada en `cv2.minAreaRect`.
2. **Acceptació condicional**: si el nombre de caràcters detectats no és entre `N_CHARS_MIN = 5` i `N_CHARS_MAX = 8`, el crop es descarta com a probable fals positiu del VJ.

**Flux de processament per crop:**
```
crop_bgr → deskew → adaptiveThreshold → findContours
        → filtre alçada+aspect ratio → validació [5-8] → resize 28×28 → guardar
```

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re
import os


def mostrar_imatge(titol, imatge, cmap=None):
    plt.figure(figsize=(10, 4))
    plt.title(titol)
    if cmap:
        plt.imshow(imatge, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(imatge, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()


# ── Directoris ────────────────────────────────────────────────────────────────
PROCESSED_DIR = Path('data/processed')   # crops del VJ: {stem}_box{n}.png
OUTPUT_DIR    = Path('data/chars')       # sortida: {stem}_box{n}_char{i}.png

# ── Rang de caràcters vàlids per a una matrícula ──────────────────────────────
N_CHARS_MIN = 5
N_CHARS_MAX = 8

# ── Mida d'entrada per a la CNN (28×28, format EMNIST-like) ──────────────────
CNN_INPUT_W = 28
CNN_INPUT_H = 28

# ── Límit de seguretat per al deskew ─────────────────────────────────────────
ALIGN_ANGLE_MAX = 15.0   # graus: si |angle| supera aquest valor no rotem

print("Llibreries carregades.")
print(f"Directori d'entrada : {PROCESSED_DIR}")
print(f"Directori de sortida: {OUTPUT_DIR}")
print(f"Rang de caràcters   : [{N_CHARS_MIN}, {N_CHARS_MAX}]")

## Pas 0: Correcció d'inclinació (deskew)

Els crops del detector Viola-Jones no estan garantidament horitzontals: el detector treballa amb una finestra rectangular però la matrícula a la foto pot estar lleugerament girada. Si apliquéssim directament el filtre d'alçada mediana sobre un crop inclinat, els caràcters tindrien bounding boxes distorsionats (massa alts o massa amples) i el filtre els rebutjaria incorrectament.

**Estratègia:** abans de qualsevol binarització, alineiem el crop:

1. Convertim a grisos i binaritzem amb Otsu (ràpid, no adaptatiu; l'únic objectiu aquí és trobar els píxels de primer pla).
2. Assegurem que els caràcters siguin **blancs** (invertim si cal).
3. `cv2.minAreaRect` sobre tots els píxels blancs: ens dóna el rectangle mínim que els conté, i el seu angle d'inclinació.
4. Convertim l'angle de `minAreaRect` a la inclinació real:
   - `minAreaRect` retorna `angle ∈ (-90°, 0°]`
   - Si `w < h` (rectangle vertical): inclinació real = `angle + 90°`
   - Si `w ≥ h` (rectangle horitzontal): inclinació real = `angle`
5. Si `|angle_corr| > ALIGN_ANGLE_MAX` no rotem: probablement la detecció és errònia.
6. Apliquem `cv2.warpAffine` amb `BORDER_REPLICATE` per evitar franges negres.

Reutilitzem l'enfocament de `char_segmenter.py` per mantenir coherència entre el notebook exploratori i el mòdul de producció.

In [ ]:
def deskew(crop_bgr):
    """
    Corregeix la inclinació d'un crop de matrícula.

    Retorna:
      aligned    : imatge BGR alineada (o l'original si no es rota)
      angle_corr : angle de correcció aplicat en graus (0.0 si no s'ha rotat)
    """
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)

    # Binaritzem amb Otsu: l'únic objectiu és localitzar els píxels de primer pla
    _, binary = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Assegurem que els caràcters siguin BLANCS (convenció de minAreaRect)
    if np.sum(binary == 255) < np.sum(binary == 0):
        binary = cv2.bitwise_not(binary)

    # Coordenades de tots els píxels de primer pla: shape (N, 2) → (row, col)
    points = np.column_stack(np.where(binary == 255))
    if len(points) < 5:
        # Massa pocs píxels: no podem estimar cap angle fiable
        return crop_bgr, 0.0

    # minAreaRect treballa amb punts (x, y) → invertim (col, row)
    rect = cv2.minAreaRect(points[:, ::-1].astype(np.float32))
    (cx, cy), (w, h), angle = rect

    # Convertim l'angle de minAreaRect ∈ (-90°, 0°] a la inclinació real:
    #   - w < h (rectangle vertical)   → la inclinació real és angle + 90°
    #   - w ≥ h (rectangle horitzontal) → la inclinació real és angle directament
    if w < h:
        angle_corr = angle + 90.0
    else:
        angle_corr = angle

    if abs(angle_corr) > ALIGN_ANGLE_MAX:
        # Angle massa gran: la detecció probablement és errònia, no rotem
        return crop_bgr, 0.0

    # Matriu de rotació al voltant del centre de la imatge
    H_img, W_img = crop_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((W_img / 2.0, H_img / 2.0), angle_corr, 1.0)

    # Rotem mantenint les mateixes dimensions; BORDER_REPLICATE evita franges negres
    aligned = cv2.warpAffine(crop_bgr, M, (W_img, H_img),
                             flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return aligned, angle_corr


# ── Demostració sobre el primer crop disponible ───────────────────────────────
crop_files = sorted(PROCESSED_DIR.glob('*_box*.png'))
if not crop_files:
    print(f"No s'han trobat fitxers a '{PROCESSED_DIR}'.")
    print("Executa primer el detector VJ o col·loca imatges {stem}_box{n}.png a data/processed/.")
else:
    example_path = crop_files[0]
    crop_bgr     = cv2.imread(str(example_path))
    aligned, angle_corr = deskew(crop_bgr)

    angle_label = f'{angle_corr:+.1f}°' if angle_corr != 0.0 else '0.0° (no calia rotar)'

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original (pre-deskew)', fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'Alineat — angle corregit: {angle_label}', fontsize=11)
    axes[1].axis('off')

    plt.suptitle(f'Pas 0: Deskew — {example_path.name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Pas 1: Adaptive Threshold

Igual que al notebook original, usem **Adaptive Threshold** en lloc d'Otsu global. La raó és la mateixa: la il·luminació dins del retall d'una matrícula real rarament és uniforme — pot haver-hi reflexos del sol, ombra del para-xocs o degradat de llum ambient. Un llindar global únic (Otsu) podria binaritzar malament les zones enfosquides o sobreexposades.

L'Adaptive Threshold calcula un llindar **diferent per a cada píxel** en funció d'un entorn local:

$$T(x,y) = \text{mean}\big(\text{entorn}(x,y)\big) - C$$

Paràmetres (idèntics al notebook de referència per mantenir compatibilitat):
- `ADAPTIVE_THRESH_MEAN_C`: la mitjana aritmètica simple dins de la finestra.
- `THRESH_BINARY_INV`: invertit perquè volem **blanc** on hi ha tinta (caràcters foscos sobre fons clar → invertit = caràcters blancs).
- `blockSize = 31`: finestra de 31×31 px — prou gran per capturar el context de cada caràcter.
- `C = 15`: offset que s'allunya prou de la mitjana per rebutjar el fons uniforme.

In [ ]:
# Continuem amb la imatge alineada del crop d'exemple

gray = cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY)

# Adaptive Threshold — paràmetres idèntics al notebook de referència
thresh = cv2.adaptiveThreshold(
    gray,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    31,   # mida de la finestra (ha de ser senar)
    15    # offset C
)

# Mostrem la imatge alineada i la seva binarització una al costat de l'altra
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
axes[0].set_title('Entrada: imatge alineada', fontsize=11)
axes[0].axis('off')

axes[1].imshow(thresh, cmap='gray')
axes[1].set_title('Sortida: Adaptive Threshold (BINARY_INV)', fontsize=11)
axes[1].axis('off')

plt.suptitle(f'Pas 1: Binarització — {example_path.name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Pas 2: Contorns externs

Repetim l'enfocament del notebook original: `cv2.findContours` sobre la màscara binaritzada. Usem:

- **`RETR_TREE`**: recuperem la jerarquia completa de contorns. Això ens permet, si cal, distingir contorns fills (forats de lletres com 'O', 'D', 'B') dels contorns pare. En el filtre posterior ja descartem la majoria de components espúries per geometria.
- **`CHAIN_APPROX_SIMPLE`**: comprimeix els segments rectes guardant només els extrems (menys memòria, mateixos bounding boxes).

En aquest pas ens limitem a **dibuixar tots els bounding boxes** en vermell per visualitzar quina quantitat de soroll s'ha generat (marges de la placa, tornillos, punts de brutícia). El pas 3 s'encarregarà de filtrar-los.

In [ ]:
# Detectem tots els contorns sobre la màscara binaritzada
cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

# Dibuixem tots els bounding boxes sobre la imatge binaritzada (convertida a RGB)
overlay_all = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
for c in cnts:
    x, y, w, h = cv2.boundingRect(c)
    cv2.rectangle(overlay_all, (x, y), (x + w, y + h), (0, 0, 255), 1)

# Mostrem la binaritzada neta i la mateixa amb tots els bboxes
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].imshow(thresh, cmap='gray')
axes[0].set_title('Binari net', fontsize=11)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(overlay_all, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Tots els contorns ({len(cnts)}) — en vermell', fontsize=11)
axes[1].axis('off')

plt.suptitle(f'Pas 2: Contorns — {example_path.name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Extreiem la llista de bounding boxes
tots_els_bboxes = [cv2.boundingRect(c) for c in cnts]
print(f"S'han detectat {len(tots_els_bboxes)} contorns en total (incloent soroll).")

## Pas 3: Filtre per alçada mediana i aspect ratio

La idea central és la mateixa que al notebook original: en una matrícula autèntica, **tots els caràcters tenen aproximadament la mateixa alçada**. Podem aprofitar-ho per distingir caràcters reals del soroll:

1. **Descartem brutícia minúscula** (`h ≤ 10 px`) per no esbiaixar la mediana.
2. **Calculem la mediana d'alçada** de la resta: és un estadístic robust (insensible als pocs outliers grans que quedin).
3. **Filtre d'alçada estricte**: acceptem els bboxes amb
   $$0.85 \cdot h_{\text{mediana}} < h < 1.15 \cdot h_{\text{mediana}}$$
   Una desviació del ±15% admet la variació natural de tipografia però rebutja marges, tornillos i fragments de la vora de la placa.
4. **Filtre d'aspect ratio**: acceptem si
   $$0.15 < \frac{w}{h} < 0.95$$
   - Límit inferior: evita línies verticals primes (soroll o vores).
   - Límit superior: evita caràcters fusionats (dues lletres segides sense separació).
5. **Ordenació d'esquerra a dreta** per `x`: garanteix que l'índex del caràcter correspon a la seva posició a la matrícula.

In [ ]:
chars_filtrats = []

# Pas 3a: mediana d'alçada (ignorem el soroll ≤ 10 px)
altures_valides = [h for (x, y, w, h) in tots_els_bboxes if h > 10]
print(f"Alçades vàlides per al càlcul de la mediana: {len(altures_valides)}")

if altures_valides:
    h_mediana = np.median(altures_valides)
    print(f"Alçada mediana de referència: {h_mediana:.1f} px")

    # Pas 3b: filtre estricte d'alçada + aspect ratio
    for (x, y, w, h) in tots_els_bboxes:
        aspect_ratio = w / float(h)
        cond_alcada = h_mediana * 0.85 < h < h_mediana * 1.15
        cond_ratio  = 0.15 < aspect_ratio < 0.95
        if cond_alcada and cond_ratio:
            chars_filtrats.append((x, y, w, h))

    # Pas 3c: ordenem d'esquerra a dreta per la coordenada x
    chars_filtrats.sort(key=lambda b: b[0])
else:
    print("No s'han trobat alçades vàlides. El crop pot ser buit o massa petit.")

# ── Visualització comparativa: tots els contorns vs. filtrats ─────────────────
# Mostra els dos estats sobre la mateixa imatge binaritzada per fer evident
# quins bboxes elimina el filtre.
overlay_all  = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
overlay_filt = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)

for (x, y, w, h) in tots_els_bboxes:
    cv2.rectangle(overlay_all, (x, y), (x + w, y + h), (0, 0, 255), 1)
for (x, y, w, h) in chars_filtrats:
    cv2.rectangle(overlay_filt, (x, y), (x + w, y + h), (0, 220, 0), 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].imshow(cv2.cvtColor(overlay_all, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Tots els contorns ({len(tots_els_bboxes)}) — vermell', fontsize=11)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(overlay_filt, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Filtrats ({len(chars_filtrats)}) — verd', fontsize=11)
axes[1].axis('off')

plt.suptitle(f'Pas 3: Filtre alçada mediana + aspect ratio — {example_path.name}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Caràcters finals retinguts: {len(chars_filtrats)}")

## Pas 4: Validació del recompte (5–8 caràcters)

El filtre geomètric del pas anterior extreu tots els candidats vàlids per geometria, però no sap res sobre el contingut semàntic del crop. Aquí afegim una **validació de negoci**: les matrícules del dataset espanyol poden tenir entre 5 i 8 caràcters (4 números + 3 lletres per a matrícules modernes, o variants antigues provincials amb menys dígits).

Si el nombre de caràcters detectats és **fora del rang `[5, 8]`**, considerem que el crop és probablement:
- Un **fals positiu del VJ** (el detector ha confos un altre rectangle de la imatge amb una matrícula).
- Una matrícula **extremadament mal segmentada** on el preprocessament no ha funcionat (soroll excessiu, il·luminació molt dolenta).

En tots dos casos, el comportament correcte és **descartar el crop sense guardar res** i continuar amb el següent. Guardar caràcters espuris contaminaria el dataset d'entrenament de la CNN.

In [ ]:
n_chars = len(chars_filtrats)

if N_CHARS_MIN <= n_chars <= N_CHARS_MAX:
    print(f"✓ Crop ACCEPTAT: {n_chars} caràcters detectats")
    accepted = True
else:
    print(f"✗ Crop REBUTJAT: {n_chars} caràcters (fora del rang [{N_CHARS_MIN}, {N_CHARS_MAX}])")
    accepted = False

## Pas 5: Copy & Resize a 28×28

Igual que al notebook original, el darrer pas de processament d'imatge és normalitzar la mida de cada caràcter:

1. **Copy**: retallem el caràcter de la imatge *threshold* (binaritzada), no de la BGR original. Les CNNs de reconeixement de text treballen millor amb text blanc sobre fons negre — és exactament el que ens dóna `THRESH_BINARY_INV`.
2. **Resize**: forcem la mida `CNN_INPUT_W × CNN_INPUT_H = 28×28` amb `INTER_AREA` (evita aliasing quan es redueix). La mida 28×28 és compatible amb EMNIST i la majoria de models de reconeixement de caràcters.

El resultat és una llista de matrius NumPy `uint8` llestes per a `model.predict(np.array(caracters_per_cnn))`.

In [ ]:
caracters_per_cnn = []

if accepted and chars_filtrats:
    for (x, y, w, h) in chars_filtrats:
        char_crop    = thresh[y:y + h, x:x + w]
        char_resized = cv2.resize(char_crop, (CNN_INPUT_W, CNN_INPUT_H),
                                  interpolation=cv2.INTER_AREA)
        caracters_per_cnn.append(char_resized)

# ── Visualització completa del pipeline per al crop d'exemple ─────────────────
# Una sola figura mostra els quatre estadis i els caràcters resultants,
# amb el mateix layout que el notebook 02_histogram_segmenter.ipynb:
#   Fila 1: original | deskew | binari + tots | binari + filtrats
#   Fila 2: caràcter 0 | caràcter 1 | ... | caràcter n-1

n_diag  = 4
n_chars_ok = len(caracters_per_cnn)
n_cols  = max(n_diag, n_chars_ok if n_chars_ok > 0 else 1)

fig = plt.figure(figsize=(max(14, n_cols * 2.2), 6))

suffix = f'  ✓ {n_chars_ok} caràcters' if accepted else '  ✗ REBUTJAT'
title_color = 'black' if accepted else 'orangered'
bbox_kw = dict(facecolor='orange', alpha=0.25, pad=4) if not accepted else {}
fig.suptitle(f'{example_path.name}{suffix}', fontsize=13, fontweight='bold',
             color=title_color, bbox=bbox_kw)

# ── Fila 1 — etapes diagnòstiques ────────────────────────────────────────────
ax1 = fig.add_subplot(2, n_cols, 1)
ax1.imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
ax1.set_title('Original', fontsize=9)
ax1.axis('off')

ax2 = fig.add_subplot(2, n_cols, 2)
ax2.imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
angle_lbl = f'Deskew ({angle_corr:+.1f}°)' if angle_corr != 0.0 else 'Deskew (0° — ok)'
ax2.set_title(angle_lbl, fontsize=9)
ax2.axis('off')

ov_all  = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
ov_filt = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
for (x, y, w, h) in tots_els_bboxes:
    cv2.rectangle(ov_all,  (x, y), (x + w, y + h), (0,   0, 255), 1)
for (x, y, w, h) in chars_filtrats:
    cv2.rectangle(ov_filt, (x, y), (x + w, y + h), (0, 220,   0), 2)

ax3 = fig.add_subplot(2, n_cols, 3)
ax3.imshow(cv2.cvtColor(ov_all, cv2.COLOR_BGR2RGB))
ax3.set_title(f'Tots els contorns ({len(tots_els_bboxes)})', fontsize=9)
ax3.axis('off')

ax4 = fig.add_subplot(2, n_cols, 4)
ax4.imshow(cv2.cvtColor(ov_filt, cv2.COLOR_BGR2RGB))
filt_color = 'green' if accepted else 'orangered'
filt_lbl   = f'Filtrats ({len(chars_filtrats)}) {"✓" if accepted else "✗"}'
ax4.set_title(filt_lbl, fontsize=9, color=filt_color)
ax4.axis('off')

# ── Fila 2 — caràcters normalitzats 28×28 ────────────────────────────────────
for i, char_img in enumerate(caracters_per_cnn):
    ax = fig.add_subplot(2, n_cols, n_cols + i + 1)
    ax.imshow(char_img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'char {i}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

if accepted:
    print(f"{len(caracters_per_cnn)} caràcters llestos per a la CNN.")
else:
    print("Crop rebutjat al pas 4 — no es processa.")

## Pas 6: Pipeline complet per a tots els crops

Ara que hem validat cada pas sobre un crop d'exemple, refactoritzem el procés en **funcions reutilitzables** i l'apliquem a tots els fitxers `*_box*.png` del directori `data/processed/`.

L'estructura del nom de fitxer `{stem}_box{box_idx}.png` és la que genera el detector VJ. El nom de sortida és `{stem}_box{box_idx}_char{i}.png` per mantenir la traçabilitat crop → caràcter i ser compatible amb l'etapa d'OCR posterior.

In [ ]:
def binarize_adaptive(aligned_bgr):
    """
    Converteix a grisos i aplica Adaptive Threshold.
    Retorna la màscara binaritzada (uint8, blanc=text, negre=fons).
    """
    gray = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        31, 15,
    )


def extract_contours(thresh):
    """
    Troba tots els contorns de la màscara binaritzada.
    Retorna la llista de bounding boxes (x, y, w, h) sense cap filtre.
    """
    cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    return [cv2.boundingRect(c) for c in cnts]


def filter_chars(bboxes):
    """
    Aplica el filtre d'alçada mediana (±15%) i aspect ratio (0.15–0.95).
    Retorna la llista filtrada, ordenada d'esquerra a dreta per x.
    """
    altures_valides = [h for (x, y, w, h) in bboxes if h > 10]
    if not altures_valides:
        return []

    h_mediana = np.median(altures_valides)
    filtrats  = [
        (x, y, w, h) for (x, y, w, h) in bboxes
        if (h_mediana * 0.85 < h < h_mediana * 1.15)
        and (0.15 < w / float(h) < 0.95)
    ]
    filtrats.sort(key=lambda b: b[0])
    return filtrats


def crops_and_resize(thresh, bboxes):
    """
    Retalla cada bbox de la màscara threshold i redimensiona a 28×28.
    Retorna una llista de np.ndarray uint8 (28×28, blanc=text).
    """
    return [
        cv2.resize(thresh[y:y + h, x:x + w], (CNN_INPUT_W, CNN_INPUT_H),
                   interpolation=cv2.INTER_AREA)
        for (x, y, w, h) in bboxes
    ]


def parse_crop_filename(path):
    """
    Extreu (stem, box_idx) de noms amb el format '{stem}_box{n}.png'.
    Retorna (None, None) si el nom no segueix el patró esperat.
    """
    m = re.match(r'^(.+)_box(\d+)$', path.stem)
    if not m:
        return None, None
    return m.group(1), int(m.group(2))


def visualize_plate(crop_bgr, aligned, angle_corr,
                    thresh, all_bboxes, filtered_bboxes,
                    chars_28, title='', accepted=True):
    """
    Figura diagnòstica completa per a un crop de matrícula.

    Layout (igual que 02_histogram_segmenter.ipynb):
      Fila 1: original | deskew+angle | binari+tots(roig) | binari+filtrats(verd)
      Fila 2: char_0 | char_1 | ... | char_n-1   (28×28 px)

    Si el crop és rebutjat (accepted=False), el títol es mostra en taronja
    i la quarta cel·la indica el motiu del rebuig.
    """
    n       = len(chars_28)
    n_diag  = 4
    n_cols  = max(n_diag, n if n > 0 else 1)

    fig = plt.figure(figsize=(max(14, n_cols * 2.2), 6))

    # Títol principal
    suffix     = f'  ✓ {n} caràcters' if accepted else '  ✗ REBUTJAT'
    title_clr  = 'black' if accepted else 'orangered'
    bbox_kw    = dict(facecolor='orange', alpha=0.25, pad=4) if not accepted else {}
    fig.suptitle(title + suffix, fontsize=12, fontweight='bold',
                 color=title_clr, bbox=bbox_kw)

    # ── Fila 1: vistes diagnòstiques ──────────────────────────────────────────

    # 1a — Crop original
    ax1 = fig.add_subplot(2, n_cols, 1)
    ax1.imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    ax1.set_title('Original', fontsize=9)
    ax1.axis('off')

    # 1b — Imatge alineada amb l'angle de correcció
    ax2 = fig.add_subplot(2, n_cols, 2)
    ax2.imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
    angle_lbl = f'Deskew ({angle_corr:+.1f}°)' if angle_corr != 0.0 else 'Deskew (0° — ok)'
    ax2.set_title(angle_lbl, fontsize=9)
    ax2.axis('off')

    # 1c — Binari amb TOTS els contorns (vermell = soroll inclòs)
    ov_all = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
    for (x, y, w, h) in all_bboxes:
        cv2.rectangle(ov_all, (x, y), (x + w, y + h), (0, 0, 255), 1)
    ax3 = fig.add_subplot(2, n_cols, 3)
    ax3.imshow(cv2.cvtColor(ov_all, cv2.COLOR_BGR2RGB))
    ax3.set_title(f'Tots els contorns ({len(all_bboxes)})', fontsize=9)
    ax3.axis('off')

    # 1d — Binari amb els caràcters FILTRATS (verd = caràcters acceptats)
    ov_filt = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
    for (x, y, w, h) in filtered_bboxes:
        cv2.rectangle(ov_filt, (x, y), (x + w, y + h), (0, 220, 0), 2)
    ax4 = fig.add_subplot(2, n_cols, 4)
    ax4.imshow(cv2.cvtColor(ov_filt, cv2.COLOR_BGR2RGB))
    filt_lbl = f'Filtrats ({len(filtered_bboxes)}) {"✓" if accepted else "✗"}'
    ax4.set_title(filt_lbl, fontsize=9, color='green' if accepted else 'orangered')
    ax4.axis('off')

    # ── Fila 2: caràcters normalitzats 28×28 ──────────────────────────────────
    for i, char_img in enumerate(chars_28):
        ax = fig.add_subplot(2, n_cols, n_cols + i + 1)
        ax.imshow(char_img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'char {i}', fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


print("Funcions definides: deskew, binarize_adaptive, extract_contours,")
print("                    filter_chars, crops_and_resize, parse_crop_filename,")
print("                    visualize_plate")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
crop_files = sorted(PROCESSED_DIR.glob('*_box*.png'))

if not crop_files:
    print(f"No s'han trobat crops a '{PROCESSED_DIR}'.")
    print("Executa primer el detector VJ per generar {stem}_box{n}.png.")
else:
    print(f"Crops trobats    : {len(crop_files)}")
    print(f"Directori sortida: {OUTPUT_DIR.resolve()}")
    print("-" * 60)

    stats = {'accepted': 0, 'rejected': 0, 'skipped': 0}

    for crop_path in crop_files:
        stem, box_idx = parse_crop_filename(crop_path)
        if stem is None:
            print(f"  ? {crop_path.name:35s} → nom inesperat (SALTAT)")
            stats['skipped'] += 1
            continue

        crop_bgr_i = cv2.imread(str(crop_path))
        if crop_bgr_i is None:
            print(f"  ! {crop_path.name:35s} → error de lectura (SALTAT)")
            stats['skipped'] += 1
            continue

        # Etapa 0: correcció d'inclinació
        aligned_i, angle_i = deskew(crop_bgr_i)

        # Etapa 1: binarització adaptativa
        thresh_i = binarize_adaptive(aligned_i)

        # Etapa 2: extracció i filtre de contorns
        all_bboxes_i = extract_contours(thresh_i)
        filtered_i   = filter_chars(all_bboxes_i)
        n_i          = len(filtered_i)
        accepted_i   = N_CHARS_MIN <= n_i <= N_CHARS_MAX

        # Etapa 3: retall + resize
        chars_28_i = crops_and_resize(thresh_i, filtered_i) if accepted_i else []

        # ── Guardar caràcters si el crop és acceptat ──────────────────────────
        if accepted_i:
            for idx, char_img in enumerate(chars_28_i):
                out_name = f'{stem}_box{box_idx}_char{idx}.png'
                cv2.imwrite(str(OUTPUT_DIR / out_name), char_img)
            angle_str = f'{angle_i:+.1f}°' if angle_i != 0.0 else ' 0.0°'
            print(f"  ✓ {crop_path.name:35s} angle={angle_str:>6}  → {n_i} chars guardats")
            stats['accepted'] += 1
        else:
            print(f"  ✗ {crop_path.name:35s} → {n_i:2d} chars (REBUTJAT)")
            stats['rejected'] += 1

        # ── Figura diagnòstica per a cada crop ───────────────────────────────
        visualize_plate(
            crop_bgr_i, aligned_i, angle_i,
            thresh_i, all_bboxes_i, filtered_i,
            chars_28_i,
            title=crop_path.name,
            accepted=accepted_i,
        )

    print("-" * 60)
    print(f"\nResum: acceptats={stats['accepted']}  rebutjats={stats['rejected']}  saltats={stats['skipped']}")
    print(f"Caràcters desats a: {OUTPUT_DIR.resolve()}")

## Visualització d'exemples

Mostrem alguns exemples de matrícules acceptades: per a cada una, carreguem la sèrie completa de caràcters `_char0...charN` des de `data/chars/` i la visualitzem en una fila de miniatures 28×28.

In [ ]:
# Cerquem fins a 4 matrícules acceptades (una per prefix diferent)
char0_files = sorted(OUTPUT_DIR.glob('*_char0.png'))[:4]

if not char0_files:
    print(f"No s'han trobat caràcters a '{OUTPUT_DIR}'.")
    print("Executa primer la cel·la del bucle principal.")
else:
    for char0_path in char0_files:
        # Reconstruïm el prefix comú (tot menys '_char0')
        prefix = re.sub(r'_char0$', '', char0_path.stem)

        # Tots els caràcters d'aquesta matrícula, ordenats per índex
        char_paths = sorted(
            OUTPUT_DIR.glob(f'{prefix}_char*.png'),
            key=lambda p: int(re.search(r'_char(\d+)', p.stem).group(1))
        )

        n = len(char_paths)
        if n == 0:
            continue

        # ── Reconstruïm el crop original i l'alineat per a la fila diagnòstica ──
        # Deduïm el nom del crop original a partir del prefix
        crop_path_i = PROCESSED_DIR / f'{prefix}.png'
        if crop_path_i.exists():
            crop_bgr_i       = cv2.imread(str(crop_path_i))
            aligned_i, ang_i = deskew(crop_bgr_i)
            thresh_i         = binarize_adaptive(aligned_i)
            all_bb_i         = extract_contours(thresh_i)
            filt_i           = filter_chars(all_bb_i)

            # Carreguem els caràcters desats
            chars_loaded = []
            for cp in char_paths:
                img = cv2.imread(str(cp), cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    chars_loaded.append(img)

            visualize_plate(
                crop_bgr_i, aligned_i, ang_i,
                thresh_i, all_bb_i, filt_i,
                chars_loaded,
                title=prefix,
                accepted=True,
            )
        else:
            # El crop original no és accessible: mostrem únicament la tira de caràcters
            fig, axes = plt.subplots(1, n, figsize=(max(6, n * 1.6), 2.5))
            if n == 1:
                axes = [axes]
            for ax, cp in zip(axes, char_paths):
                img = cv2.imread(str(cp), cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
                ax.set_title(re.search(r'char(\d+)', cp.stem).group(0), fontsize=8)
                ax.axis('off')
            plt.suptitle(prefix, fontsize=10, fontweight='bold')
            plt.tight_layout()
            plt.show()